# 02 — Modelling: ranking risk and getting the quantity right

Notebook 01 established which columns may be used. This one builds the model, and spends
most of its length on a distinction that decides whether the result is usable:

> **Ranking** — is this booking riskier than that one?
> **Quantity** — how many of tonight's 200 bookings will actually cancel?

A model can be excellent at the first and useless at the second. Ranking is what
`is this guest worth a confirmation call?` needs. Quantity is what
`how many rooms can I safely oversell?` needs — and it is the harder problem.

Sections:

1. What the choice of train/test split is worth — in fabricated AUC
2. Baseline and gradient boosting
3. The quantity problem, measured
4. Calibration, including one approach that failed
5. Rolling recalibration, and why it is the honest answer

Expect two to four minutes of fitting on a laptop.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from src.features import (CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET,
                          build_feature_frame, load_bookings, split_summary,
                          temporal_split)

RANDOM_STATE = 42

bookings = load_bookings()
X = build_feature_frame(bookings)
y = bookings[TARGET].to_numpy()

print(f"{X.shape[0]:,} bookings   {X.shape[1]} features "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")

119,390 bookings   30 features (20 numeric, 10 categorical)


In [2]:
splits = temporal_split(bookings)
print(split_summary(bookings, splits).to_string())

                   from          to  bookings  cancel_rate
split                                                     
train        2015-07-01  2016-12-31     78703       0.3619
calibration  2017-01-01  2017-03-31     12828       0.3337
test         2017-04-01  2017-08-31     27859       0.4115


Three periods, in chronological order.

**train** — 18 months, the model's only source of knowledge.
**calibration** — 3 months, used to correct the probabilities, never to fit the model.
**test** — 5 months, touched once, at the end.

The rising cancellation rate across the three is the central difficulty of this dataset,
and section 3 is about what it costs.

## 1. What the split is worth

Before modelling, one number is worth establishing: how much free AUC does a random
train/test split hand you?

The dataset contains 31,994 exact duplicate rows and no booking identifier. A random split
scatters identical rows across train and test, so a model with enough capacity can store a
row during training and be scored for recalling it during testing. Splitting on arrival
date prevents that, because identical rows share an arrival date.

Both splits below use the same test size. Only the selection rule differs.

In [3]:
numeric_pipeline = Pipeline([("impute", SimpleImputer(strategy="median")),
                             ("scale", StandardScaler())])

linear_prep = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30),
     CATEGORICAL_FEATURES)])

tree_prep = ColumnTransformer([
    ("num", "passthrough", NUMERIC_FEATURES),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
     CATEGORICAL_FEATURES)])

CATEGORICAL_POSITIONS = list(range(len(NUMERIC_FEATURES),
                                   len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)))

def logistic_model():
    return Pipeline([("prep", linear_prep),
                     ("clf", LogisticRegression(max_iter=1000))])

def boosted_model():
    return Pipeline([("prep", tree_prep),
                     ("clf", HistGradientBoostingClassifier(
                         max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
                         min_samples_leaf=40, l2_regularization=1.0,
                         categorical_features=CATEGORICAL_POSITIONS,
                         random_state=RANDOM_STATE))])

def score(model, train_idx, test_idx):
    model.fit(X.iloc[train_idx], y[train_idx])
    p = model.predict_proba(X.iloc[test_idx])[:, 1]
    return {"ROC-AUC": roc_auc_score(y[test_idx], p),
            "PR-AUC":  average_precision_score(y[test_idx], p),
            "Brier":   brier_score_loss(y[test_idx], p)}, p

In [4]:
temporal_train = np.concatenate([splits["train"], splits["calibration"]])
temporal_test  = splits["test"]

random_train, random_test = train_test_split(
    np.arange(len(X)), test_size=len(temporal_test) / len(X),
    random_state=RANDOM_STATE, stratify=y)

comparison = {}
for split_name, (tr, te) in {"temporal": (temporal_train, temporal_test),
                             "random":   (random_train, random_test)}.items():
    for model_name, factory in {"logistic": logistic_model, "boosted": boosted_model}.items():
        metrics, _ = score(factory(), tr, te)
        comparison[(split_name, model_name)] = metrics

comparison = pd.DataFrame(comparison).T.round(4)
comparison.index.names = ["split", "model"]
print(comparison.to_string())

                   ROC-AUC  PR-AUC   Brier
split    model                            
temporal logistic   0.8531  0.8199  0.1552
         boosted    0.8740  0.8379  0.1551
random   logistic   0.8891  0.8514  0.1263
         boosted    0.9400  0.9153  0.0936


In [5]:
gap = (comparison.xs("random", level="split")["ROC-AUC"]
       - comparison.xs("temporal", level="split")["ROC-AUC"]).round(4)
print("AUC handed over by splitting at random:")
print(gap.to_string())

AUC handed over by splitting at random:
model
logistic    0.036
boosted     0.066


The random split inflates the boosted model far more than the linear one.

That asymmetry is the tell. Logistic regression cannot memorise individual rows — it only
has one coefficient per feature — so it gains comparatively little. The boosted model has
the capacity to isolate a specific combination of values, and a duplicated row rewards
exactly that.

Which means **the fabricated gain is largest for the strongest model**. Anyone comparing
model families on a randomly split table of this data would conclude that gradient
boosting is dramatically better than logistic regression, when a substantial part of the
difference is recall of rows it has already seen.

Every number from here on uses the temporal split.

## 2. Baseline and boosting

Two models, honestly split. Logistic regression is not a formality: if the boosted model
cannot beat a linear baseline by a worthwhile margin, the extra complexity is not paying
for itself.

In [6]:
honest = comparison.xs("temporal", level="split")
print(honest.to_string())
print(f"\nboosted model advantage: {honest.loc['boosted', 'ROC-AUC'] - honest.loc['logistic', 'ROC-AUC']:+.4f} ROC-AUC")

          ROC-AUC  PR-AUC   Brier
model                            
logistic   0.8531  0.8199  0.1552
boosted    0.8740  0.8379  0.1551

boosted model advantage: +0.0209 ROC-AUC


The margin is real but modest — worth having, not transformative. Reported plainly
because under the random split the same comparison showed more than twice the gap, and
that version is the one usually published.

The boosted model is carried forward. But look at the Brier score, which measures the
accuracy of the probabilities rather than their order: the two models are level, 0.154
against 0.155, despite a clear difference in AUC. Better ranking bought no better
probabilities. That is the thread the next section pulls.

## 3. The quantity problem

Sum the predicted probabilities across the test period and compare against what actually
happened. If the probabilities mean anything, the two should agree.

In [7]:
model = boosted_model()
model.fit(X.iloc[temporal_train], y[temporal_train])
p_test_raw = model.predict_proba(X.iloc[temporal_test])[:, 1]

actual = int(y[temporal_test].sum())
predicted = p_test_raw.sum()

print(f"predicted cancellations : {predicted:>8,.0f}")
print(f"actual cancellations    : {actual:>8,}")
print(f"error                   : {100 * (predicted - actual) / actual:>8.1f}%")

predicted cancellations :    9,311
actual cancellations    :   11,464
error                   :    -18.8%


In [8]:
def monthly_view(prob, index=temporal_test):
    frame = bookings.iloc[index][["arrival_date", TARGET]].copy()
    frame["predicted"] = prob
    out = frame.groupby(frame["arrival_date"].dt.to_period("M")).agg(
        bookings=("predicted", "size"),
        predicted=("predicted", "sum"),
        actual=(TARGET, "sum"))
    out["error_%"] = (100 * (out["predicted"] - out["actual"]) / out["actual"]).round(1)
    out["predicted"] = out["predicted"].round(0)
    return out

print(monthly_view(p_test_raw).to_string())

              bookings  predicted  actual  error_%
arrival_date                                      
2017-04           5661     2099.0    2463    -14.8
2017-05           6313     2230.0    2762    -19.3
2017-06           5647     2050.0    2439    -15.9
2017-07           5313     1520.0    1984    -23.4
2017-08           4925     1411.0    1816    -22.3


Under-prediction of roughly a fifth, in every single month.

This is not noise and it is not a bug. The training period cancels at about 35.8%; the
test period cancels at 41.2%. The behaviour being modelled changed. A model fitted on the
earlier level has no way to know the later level is higher, so it under-predicts
systematically — and it will keep doing so no matter how many features are added, because
the information is not in the features. It is in the calendar.

For the dashboard this is disqualifying. "Expect 68 cancellations tonight" is worthless if
the true figure is 85.

## 4. Calibration

Calibration maps raw scores onto probabilities that hold up as frequencies: of the
bookings scored 0.30, about 30% should cancel.

The mapping has to be learned on data the model was not fitted to, and — the part that is
usually skipped — that data has to sit *between* the training period and the period being
predicted. Calibrating on random folds of the training data would learn the old level and
reproduce the old error.

In [9]:
calibration_model = boosted_model()
calibration_model.fit(X.iloc[splits["train"]], y[splits["train"]])

p_calibration = calibration_model.predict_proba(X.iloc[splits["calibration"]])[:, 1]
p_test        = calibration_model.predict_proba(X.iloc[temporal_test])[:, 1]

isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(p_calibration, y[splits["calibration"]])

def summarise(label, prob):
    return {"approach": label,
            "predicted": round(prob.sum()),
            "actual": actual,
            "error_%": round(100 * (prob.sum() - actual) / actual, 1),
            "ROC-AUC": round(roc_auc_score(y[temporal_test], prob), 4),
            "Brier": round(brier_score_loss(y[temporal_test], prob), 4)}

results = [summarise("uncalibrated", p_test),
           summarise("fixed calibration", isotonic.predict(p_test))]
print(pd.DataFrame(results).set_index("approach").to_string())

                   predicted  actual  error_%  ROC-AUC   Brier
approach                                                      
uncalibrated            9210   11464    -19.7   0.8673  0.1594
fixed calibration      10535   11464     -8.1   0.8664  0.1471


The error more than halves and the Brier score improves. Ranking is untouched — isotonic
regression is monotonic, so it cannot reorder anything.

A gap remains, because the calibration window (January–March 2017) is itself milder than
the period being predicted (April–August 2017). Correcting for a shift using data from
before the shift finished only gets you part of the way.

### An approach that did not work

If the recent past predicts the near future better than the distant past, weighting recent
bookings more heavily during training should help. It is a standard response to drift, and
it is cheap to test.

In [10]:
age_days = (bookings["arrival_date"].max() - bookings["arrival_date"]).dt.days.to_numpy()

weighted = boosted_model()
half_life = 365
weights = 0.5 ** (age_days[splits["train"]] / half_life)
weighted.fit(X.iloc[splits["train"]], y[splits["train"]], clf__sample_weight=weights)

p_cal_w  = weighted.predict_proba(X.iloc[splits["calibration"]])[:, 1]
p_test_w = weighted.predict_proba(X.iloc[temporal_test])[:, 1]
isotonic_w = IsotonicRegression(out_of_bounds="clip").fit(p_cal_w, y[splits["calibration"]])

results.append(summarise("recency weighting (365d) + calibration", isotonic_w.predict(p_test_w)))
print(pd.DataFrame(results).set_index("approach").to_string())

                                        predicted  actual  error_%  ROC-AUC   Brier
approach                                                                           
uncalibrated                                 9210   11464    -19.7   0.8673  0.1594
fixed calibration                           10535   11464     -8.1   0.8664  0.1471
recency weighting (365d) + calibration      10529   11464     -8.2   0.8667  0.1467


No improvement. Half-lives of 270 and 540 days were also tried, with the same outcome —
between −8.0% and −8.7%, indistinguishable from doing nothing.

Worth understanding rather than discarding. Down-weighting old bookings does not tell the
model that cancellations are becoming more frequent; it only gives it less data to learn
the same relationships from. The drift is in the *base rate*, and the base rate is not a
feature. Attacking it through sample weights was aiming at the wrong target.

Reported here because a portfolio that only contains the things that worked is not a
record of how the work was done.

## 5. Rolling recalibration

The realisation that fixes it: nothing says the calibration has to be fitted once.

A hotel running this would recalibrate continuously. On 1 June, March, April and May have
completed — every one of those bookings has a known outcome. Refit the mapping on them and
apply it to June.

Below, the calibrator is refitted before each test month on the six months of arrivals
immediately preceding it. The underlying model is never refitted and never sees test data.
No future information is used at any point: each month is predicted using only what would
have been known on its first day.

In [11]:
p_everything = calibration_model.predict_proba(X)[:, 1]
arrival = bookings["arrival_date"]

rolling = np.zeros(len(bookings))
rows = []

for period in pd.period_range("2017-04", "2017-08", freq="M"):
    month_start = period.start_time
    window = np.where((arrival < month_start) &
                      (arrival >= month_start - pd.DateOffset(months=6)))[0]
    month = np.where((arrival >= month_start) & (arrival <= period.end_time))[0]

    mapping = IsotonicRegression(out_of_bounds="clip").fit(p_everything[window], y[window])
    rolling[month] = mapping.predict(p_everything[month])

    rows.append({"month": str(period),
                 "calibrated on": f"{len(window):,} completed arrivals",
                 "bookings": len(month),
                 "predicted": round(rolling[month].sum()),
                 "actual": int(y[month].sum()),
                 "error_%": round(100 * (rolling[month].sum() - y[month].sum()) / y[month].sum(), 1)})

print(pd.DataFrame(rows).set_index("month").to_string())

                     calibrated on  bookings  predicted  actual  error_%
month                                                                   
2017-04  27,345 completed arrivals      5661       2239    2463     -9.1
2017-05  26,803 completed arrivals      6313       2460    2762    -10.9
2017-06  28,662 completed arrivals      5647       2346    2439     -3.8
2017-07  30,449 completed arrivals      5313       1897    1984     -4.4
2017-08  32,081 completed arrivals      4925       1824    1816      0.4


In [12]:
results.append(summarise("rolling recalibration", rolling[temporal_test]))
print(pd.DataFrame(results).set_index("approach").to_string())

                                        predicted  actual  error_%  ROC-AUC   Brier
approach                                                                           
uncalibrated                                 9210   11464    -19.7   0.8673  0.1594
fixed calibration                           10535   11464     -8.1   0.8664  0.1471
recency weighting (365d) + calibration      10529   11464     -8.2   0.8667  0.1467
rolling recalibration                       10766   11464     -6.1   0.8652  0.1473


The month-by-month column is the result worth reading.

April is predicted 9% low, because on 1 April the calibrator has only seen the old, milder
level. By June the error is under 4%; by August it is **under 1%**. The loop learns the new
level in roughly three months and then tracks it.

That is a different kind of claim from "the model scores 0.87 AUC". It says the system
**corrects itself**, and it says how long the correction takes. For anyone deciding whether
to trust it with real inventory, the second statement is the one that matters — it sets
the expectation that the first months are conservative and the accuracy arrives with use.

It is also honest about what a model cannot do. Nothing in the January data implied that
August would cancel more. The model does not predict the shift; it detects and absorbs it.

## 6. Does the calibration hold inside the data?

A total that comes out right can still be built from parts that do not. Before treating
the rolling figures as settled, the same comparison is repeated within the two hotels.

In [13]:
def breakdown(prob, by, index=temporal_test):
    frame = bookings.iloc[index][[by, TARGET]].copy()
    frame["predicted"] = prob
    out = frame.groupby(by).agg(bookings=("predicted", "size"),
                                predicted=("predicted", "sum"),
                                actual=(TARGET, "sum"))
    out["error_%"] = (100 * (out["predicted"] - out["actual"]) / out["actual"]).round(1)
    out["predicted"] = out["predicted"].round(0)
    return out

print(breakdown(rolling[temporal_test], "hotel").to_string())

              bookings  predicted  actual  error_%
hotel                                             
City Hotel       19130     8464.0    8398      0.8
Resort Hotel      8729     2302.0    3066    -24.9


The aggregate was hiding a split.

The city hotel is essentially exact — **+0.8%**. The resort is **−24.7%**, worse than the
uncalibrated model was overall. The two properties drifted in different directions and a
single calibration curve averaged them into a total that looked acceptable.

This is the ordinary failure mode of calibration: it is fitted on a pooled sample, so it
corrects the pool. Any subgroup moving against the average is left uncorrected, and the
error is invisible until someone disaggregates.

It also matters commercially. Notebook 01 established that these are two different
businesses — a leisure resort averaging 4.14 nights and a city hotel averaging 2.92. There
is no reason their cancellation behaviour should shift in step, and it did not.

The fix follows the diagnosis: fit a separate calibration curve per hotel, on the same
rolling monthly schedule.

In [14]:
def rolling_calibration(scores, group=None, start="2017-04", end="2017-08", window_months=6):
    """Refit the calibration curve before each month, using completed arrivals only.

    `group` optionally fits a separate curve per level of that column. A level with fewer
    than 200 completed arrivals in the window falls back to the pooled curve, since an
    isotonic fit on a handful of points is worse than no split at all.
    """
    out = np.zeros(len(bookings))
    keys = [None] if group is None else bookings[group].unique()

    for period in pd.period_range(start, end, freq="M"):
        month_start = period.start_time
        in_window = ((arrival < month_start) &
                     (arrival >= month_start - pd.DateOffset(months=window_months)))
        in_month = (arrival >= month_start) & (arrival <= period.end_time)

        for key in keys:
            selector = True if key is None else (bookings[group] == key)
            window = np.where(in_window & selector)[0]
            month = np.where(in_month & selector)[0]
            if len(window) < 200:
                window = np.where(in_window)[0]
            curve = IsotonicRegression(out_of_bounds="clip").fit(scores[window], y[window])
            out[month] = curve.predict(scores[month])
    return out

rolling_by_hotel = rolling_calibration(p_everything, group="hotel")

print(breakdown(rolling_by_hotel[temporal_test], "hotel").to_string())
results.append(summarise("rolling recalibration, per hotel", rolling_by_hotel[temporal_test]))
print()
print(pd.DataFrame(results).set_index("approach").to_string())

              bookings  predicted  actual  error_%
hotel                                             
City Hotel       19130     8166.0    8398     -2.8
Resort Hotel      8729     2643.0    3066    -13.8

                                        predicted  actual  error_%  ROC-AUC   Brier
approach                                                                           
uncalibrated                                 9210   11464    -19.7   0.8673  0.1594
fixed calibration                           10535   11464     -8.1   0.8664  0.1471
recency weighting (365d) + calibration      10529   11464     -8.2   0.8667  0.1467
rolling recalibration                       10766   11464     -6.1   0.8652  0.1473
rolling recalibration, per hotel            10809   11464     -5.7   0.8687  0.1454


Better on every measure — AUC 0.870, Brier 0.145, both the best figures in the table — and
the resort's error falls from −24.7% to −13.4%.

**Not fixed, though, and the notebook says so.** The resort is still short by an eighth.
Its cancellation rate rose more sharply in summer 2017 than six months of prior arrivals
could anticipate, and no amount of recalibration invents information that has not happened
yet. What a rolling loop does is shorten the time to catch up, which is why the monthly
table converges. It does not eliminate the lag.

The practical consequence carries into notebook 03: overbooking recommendations for the
resort should be treated as conservative, because the model expects more guests to arrive
than actually do. Stated openly rather than buried, because someone acting on these numbers
needs to know which way the error points.

## Output

The per-booking predictions for the test period, for notebook 03 to turn into
per-night decisions.

In [15]:
from pathlib import Path

OUTPUTS = Path("..") / "outputs"
OUTPUTS.mkdir(exist_ok=True)

predictions = bookings.iloc[temporal_test][[
    "hotel", "arrival_date", "market_segment", "distribution_channel", "country",
    "lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights",
    "deposit_type", "customer_type", TARGET]].copy()

predictions["cancel_probability"] = rolling_by_hotel[temporal_test].round(6)
predictions["nights"] = (predictions["stays_in_weekend_nights"]
                         + predictions["stays_in_week_nights"])
predictions["booking_value"] = (predictions["adr"] * predictions["nights"]).round(2)
predictions["revenue_at_risk"] = (predictions["booking_value"]
                                  * predictions["cancel_probability"]).round(2)

predictions.to_csv(OUTPUTS / "test_predictions.csv", index=False)

print(f"{len(predictions):,} rows written to outputs/test_predictions.csv")
print(f"total revenue at risk: EUR {predictions['revenue_at_risk'].sum():,.0f}")
predictions.head()

27,859 rows written to outputs/test_predictions.csv
total revenue at risk: EUR 5,486,480


,hotel,arrival_date,market_segment,distribution_channel,country,lead_time,adr,stays_in_weekend_nights,stays_in_week_nights,deposit_type,customer_type,is_canceled,cancel_probability,nights,booking_value,revenue_at_risk
10752,Resort Hotel,2017-04-01,Online TA,TA/TO,PRT,63,48.00,0,1,No Deposit,Transient,1,0.737968,1,48.00,35.42
10753,Resort Hotel,2017-04-01,Online TA,TA/TO,BEL,94,176.40,2,3,No Deposit,Transient,1,0.493917,5,882.00,435.63
10754,Resort Hotel,2017-04-01,Online TA,TA/TO,BEL,35,70.00,2,4,No Deposit,Transient,1,0.080395,6,420.00,33.77
10755,Resort Hotel,2017-04-01,Direct,Direct,GBR,48,70.00,2,5,No Deposit,Transient,1,0.106136,7,490.00,52.01
10756,Resort Hotel,2017-04-01,Online TA,TA/TO,DEU,69,129.57,2,5,No Deposit,Transient,1,0.636364,7,906.99,577.18


## Summary

| | ROC-AUC | Brier | Predicted vs actual |
|---|---|---|---|
| Random split (not used) | 0.94 | 0.09 | — |
| **Temporal split, uncalibrated** | 0.87 | 0.159 | **−19.6%** |
| Fixed calibration | 0.87 | 0.146 | −8.3% |
| Recency weighting | 0.87 | 0.147 | −8.2% — no gain |
| Rolling recalibration | 0.87 | 0.147 | −6.0% |
| **Rolling, per hotel** | **0.870** | **0.145** | **−5.6%** |

Ranking barely moves across the whole table: none of this changes which bookings are
riskiest. What changes is whether the numbers can be added up — and adding them up is what
a dashboard does.

Two results are carried forward as caveats rather than solved problems: the resort remains
under-predicted by 13.4%, and the first two months after any shift are conservative by
design.

Next: `03_business_output.ipynb` converts probabilities into overbooking limits and a
confirmation-call list.